# 📅 Módulo 3 — Control Diario
**Sistema de Gestión de Recursos Humanos**

Este módulo gestiona el control diario del personal activo:
- **Asistencia** → registro de presencia/falta por día
- **Ausencias** → falta justificada con motivo
- **Vacaciones** → solicitud y registro de períodos de vacaciones

> **Base de datos:** `rrhh_sistema` · **Puerto:** `1989` · **Motor:** MySQL


In [ ]:
# ── 1. Instalación automática de dependencias ─────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'mysql-connector-python', 'ipywidgets', 'pandas', '--quiet'], check=False)

import mysql.connector
from mysql.connector import Error
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
from datetime import date, timedelta

print('✅ Librerías cargadas correctamente.')


In [ ]:
# Conexion a la base de datos
import os
import mysql.connector

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', '1989')),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'rrhh_sistema'),
    'charset': 'utf8mb4',
}

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

try:
    conn = get_conn()
    conn.close()
    print(f'Conexion exitosa -> {DB_CONFIG["database"]} (Puerto {DB_CONFIG["port"]})')
except mysql.connector.Error as e:
    print(f'Error de conexion: {e}')



In [ ]:
# ── 3. Funciones auxiliares ───────────────────────────────────────────────

def get_empleados_activos() -> list:
    """Devuelve [(codigo_empresa, nombre_completo), ...] de empleados activos."""
    conn = get_conn(); cur = conn.cursor()
    cur.execute("""
        SELECT e.codigo_empresa,
               CONCAT(e.nombre, ' ', e.apellido, '  [', e.codigo_empresa, ']') AS label
        FROM   empleados e
        WHERE  e.estado = 'activo'
        ORDER  BY e.apellido, e.nombre
    """)
    rows = cur.fetchall(); cur.close(); conn.close()
    return rows

def registrar_asistencia(codigo: str, fecha, presente: bool) -> str:
    """Inserta o actualiza el registro de asistencia de un día."""
    conn = get_conn(); cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO asistencias (codigo_empresa, fecha, presente)
            VALUES (%s, %s, %s)
            ON DUPLICATE KEY UPDATE presente = VALUES(presente)
        """, (codigo, fecha, 1 if presente else 0))
        conn.commit()
        estado = 'Presente ✅' if presente else 'Falta ❌'
        return f'✅ Asistencia registrada: {codigo} — {fecha} — {estado}'
    except Error as e:
        conn.rollback(); return f'❌ Error: {e}'
    finally:
        cur.close(); conn.close()

def registrar_ausencia(codigo: str, fecha, motivo: str) -> str:
    """Registra una ausencia justificada."""
    conn = get_conn(); cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO ausencias (codigo_empresa, fecha, motivo)
            VALUES (%s, %s, %s)
            ON DUPLICATE KEY UPDATE motivo = VALUES(motivo)
        """, (codigo, fecha, motivo or None))
        # también marca como no-presente en asistencias
        cur.execute("""
            INSERT INTO asistencias (codigo_empresa, fecha, presente)
            VALUES (%s, %s, 0)
            ON DUPLICATE KEY UPDATE presente = 0
        """, (codigo, fecha))
        conn.commit()
        return f'✅ Ausencia registrada: {codigo} — {fecha}'
    except Error as e:
        conn.rollback(); return f'❌ Error: {e}'
    finally:
        cur.close(); conn.close()

def registrar_vacaciones(codigo: str, fecha_inicio, fecha_fin, observaciones: str) -> str:
    """Registra un período de vacaciones."""
    if fecha_fin < fecha_inicio:
        return '⚠️ La fecha de fin debe ser igual o posterior a la fecha de inicio.'
    conn = get_conn(); cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO vacaciones (codigo_empresa, fecha_inicio, fecha_fin, observaciones)
            VALUES (%s, %s, %s, %s)
        """, (codigo, fecha_inicio, fecha_fin, observaciones or None))
        conn.commit()
        dias = (fecha_fin - fecha_inicio).days + 1
        return f'✅ Vacaciones registradas: {codigo} — {fecha_inicio} al {fecha_fin} ({dias} días)'
    except Error as e:
        conn.rollback(); return f'❌ Error: {e}'
    finally:
        cur.close(); conn.close()

print('✅ Funciones auxiliares cargadas.')


In [ ]:
# ── 4. PANEL PRINCIPAL CON PESTAÑAS ──────────────────────────────────────

STYLE  = {'description_width': '160px'}
LAYOUT = widgets.Layout(width='420px')
BTN_LY = widgets.Layout(width='220px', margin='8px 4px 0 0')
OUT_LY = widgets.Layout(margin='8px 0 0 0', border='1px solid #e0e0e0',
                         padding='10px', border_radius='6px', min_height='40px')

# Cargar empleados activos para dropdowns
emp_raw  = get_empleados_activos()
emp_opts = {label: codigo for codigo, label in emp_raw}

def emp_dropdown(desc='Empleado *'):
    return widgets.Dropdown(description=desc, options=list(emp_opts.keys()),
                             style=STYLE, layout=LAYOUT)

# ══════════════════════════════════════════════════════════════════════════
# TAB 1 — ASISTENCIA
# ══════════════════════════════════════════════════════════════════════════
w_a_emp    = emp_dropdown('Empleado *')
w_a_fecha  = widgets.DatePicker(description='Fecha *', value=date.today(),
                                 style=STYLE, layout=LAYOUT)
w_a_estado = widgets.RadioButtons(description='Estado *',
                                   options=['Presente', 'Falta'],
                                   style=STYLE, layout=LAYOUT)
btn_a_guardar = widgets.Button(description='💾 Guardar Asistencia',
                                button_style='success', layout=BTN_LY)
out_a = widgets.Output(layout=OUT_LY)

btn_a_ver     = widgets.Button(description='📋 Ver registros del día',
                                button_style='info', layout=BTN_LY)
out_a_ver = widgets.Output(layout=OUT_LY)

def on_guardar_asistencia(b):
    with out_a:
        clear_output()
        if not emp_opts: print('⚠️ No hay empleados activos.'); return
        codigo  = emp_opts[w_a_emp.value]
        presente = w_a_estado.value == 'Presente'
        print(registrar_asistencia(codigo, w_a_fecha.value, presente))

def on_ver_asistencia(b):
    with out_a_ver:
        clear_output()
        try:
            conn = get_conn()
            df = pd.read_sql("""
                SELECT e.codigo_empresa, CONCAT(e.nombre,' ',e.apellido) AS empleado,
                       a.fecha,
                       IF(a.presente, 'Presente ✅', 'Falta ❌') AS estado
                FROM   asistencias a
                JOIN   empleados e ON e.codigo_empresa = a.codigo_empresa
                WHERE  a.fecha = CURDATE()
                ORDER  BY e.apellido
            """, conn); conn.close()
            if df.empty: print('No hay registros de asistencia para hoy.')
            else:
                display(HTML("<b>Asistencia de hoy</b>"))
                display(df.style.hide(axis='index'))
        except Exception as ex:
            print(f'❌ Error: {ex}')

btn_a_guardar.on_click(on_guardar_asistencia)
btn_a_ver.on_click(on_ver_asistencia)

tab1 = widgets.VBox([
    widgets.HTML("<h3 style='color:#1a73e8'>📌 Registrar Asistencia</h3>"),
    w_a_emp, w_a_fecha, w_a_estado,
    widgets.HBox([btn_a_guardar]), out_a,
    widgets.HTML("<hr>"),
    widgets.HBox([btn_a_ver]), out_a_ver
], layout=widgets.Layout(padding='16px'))

# ══════════════════════════════════════════════════════════════════════════
# TAB 2 — AUSENCIAS
# ══════════════════════════════════════════════════════════════════════════
w_au_emp    = emp_dropdown('Empleado *')
w_au_fecha  = widgets.DatePicker(description='Fecha ausencia *', value=date.today(),
                                  style=STYLE, layout=LAYOUT)
w_au_motivo = widgets.Text(description='Motivo *',
                            placeholder='Ej. Cita médica, Permiso personal...',
                            style=STYLE, layout=LAYOUT)
btn_au_guardar = widgets.Button(description='💾 Registrar Ausencia',
                                 button_style='warning', layout=BTN_LY)
out_au = widgets.Output(layout=OUT_LY)

btn_au_ver = widgets.Button(description='📋 Ver ausencias del mes',
                             button_style='info', layout=BTN_LY)
out_au_ver = widgets.Output(layout=OUT_LY)

def on_guardar_ausencia(b):
    with out_au:
        clear_output()
        if not emp_opts: print('⚠️ No hay empleados activos.'); return
        if not w_au_motivo.value.strip():
            print('⚠️ El motivo es requerido.'); return
        codigo = emp_opts[w_au_emp.value]
        print(registrar_ausencia(codigo, w_au_fecha.value, w_au_motivo.value.strip()))

def on_ver_ausencias(b):
    with out_au_ver:
        clear_output()
        try:
            conn = get_conn()
            df = pd.read_sql("""
                SELECT CONCAT(e.nombre,' ',e.apellido) AS empleado,
                       au.fecha, au.motivo
                FROM   ausencias au
                JOIN   empleados e ON e.codigo_empresa = au.codigo_empresa
                WHERE  MONTH(au.fecha) = MONTH(CURDATE())
                  AND  YEAR(au.fecha)  = YEAR(CURDATE())
                ORDER  BY au.fecha DESC
            """, conn); conn.close()
            if df.empty: print('No hay ausencias registradas este mes.')
            else:
                display(HTML("<b>Ausencias del mes actual</b>"))
                display(df.style.hide(axis='index'))
        except Exception as ex:
            print(f'❌ Error: {ex}')

btn_au_guardar.on_click(on_guardar_ausencia)
btn_au_ver.on_click(on_ver_ausencias)

tab2 = widgets.VBox([
    widgets.HTML("<h3 style='color:#f9a825'>🟡 Registrar Ausencia Justificada</h3>"),
    w_au_emp, w_au_fecha, w_au_motivo,
    widgets.HBox([btn_au_guardar]), out_au,
    widgets.HTML("<hr>"),
    widgets.HBox([btn_au_ver]), out_au_ver
], layout=widgets.Layout(padding='16px'))

# ══════════════════════════════════════════════════════════════════════════
# TAB 3 — VACACIONES
# ══════════════════════════════════════════════════════════════════════════
w_v_emp    = emp_dropdown('Empleado *')
w_v_inicio = widgets.DatePicker(description='Fecha inicio *', value=date.today(),
                                 style=STYLE, layout=LAYOUT)
w_v_fin    = widgets.DatePicker(description='Fecha fin *',
                                 value=date.today() + timedelta(days=6),
                                 style=STYLE, layout=LAYOUT)
w_v_obs    = widgets.Textarea(description='Observaciones',
                               placeholder='(opcional)',
                               style=STYLE, layout=widgets.Layout(width='420px'), rows=2)
btn_v_guardar = widgets.Button(description='💾 Registrar Vacaciones',
                                button_style='success', layout=BTN_LY)
out_v = widgets.Output(layout=OUT_LY)

btn_v_ver  = widgets.Button(description='📋 Ver vacaciones activas',
                             button_style='info', layout=BTN_LY)
out_v_ver  = widgets.Output(layout=OUT_LY)

def on_guardar_vacaciones(b):
    with out_v:
        clear_output()
        if not emp_opts: print('⚠️ No hay empleados activos.'); return
        codigo = emp_opts[w_v_emp.value]
        print(registrar_vacaciones(codigo, w_v_inicio.value, w_v_fin.value, w_v_obs.value.strip()))

def on_ver_vacaciones(b):
    with out_v_ver:
        clear_output()
        try:
            conn = get_conn()
            df = pd.read_sql("""
                SELECT CONCAT(e.nombre,' ',e.apellido) AS empleado,
                       v.fecha_inicio, v.fecha_fin, v.dias_tomados, v.observaciones
                FROM   vacaciones v
                JOIN   empleados e ON e.codigo_empresa = v.codigo_empresa
                WHERE  v.fecha_fin >= CURDATE()
                ORDER  BY v.fecha_inicio
            """, conn); conn.close()
            if df.empty: print('No hay vacaciones activas o próximas.')
            else:
                display(HTML("<b>Vacaciones activas / próximas</b>"))
                display(df.style.hide(axis='index'))
        except Exception as ex:
            print(f'❌ Error: {ex}')

btn_v_guardar.on_click(on_guardar_vacaciones)
btn_v_ver.on_click(on_ver_vacaciones)

tab3 = widgets.VBox([
    widgets.HTML("<h3 style='color:#2e7d32'>🌴 Solicitud de Vacaciones</h3>"),
    w_v_emp, w_v_inicio, w_v_fin, w_v_obs,
    widgets.HBox([btn_v_guardar]), out_v,
    widgets.HTML("<hr>"),
    widgets.HBox([btn_v_ver]), out_v_ver
], layout=widgets.Layout(padding='16px'))

# ══════════════════════════════════════════════════════════════════════════
# PANEL FINAL
# ══════════════════════════════════════════════════════════════════════════
panel = widgets.Tab(children=[tab1, tab2, tab3])
panel.set_title(0, '📌 Asistencia')
panel.set_title(1, '🟡 Ausencias')
panel.set_title(2, '🌴 Vacaciones')

display(panel)
